# Agentic RAG Pipeline with Google Gemini 3.7 Flash

This notebook provides a complete, clean end-to-end RAG pipeline using Google Gemini (`google-genai` SDK), `minsearch`, and structured prompt engineering.

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from minsearch import Index

# Load API Keys from .env
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set in your .env file.")

# Initialize Gemini Client
client = genai.Client(api_key=gemini_api_key)
print("Gemini Client initialized successfully!")

### 1. Test Basic Gemini Generation

In [ ]:
response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents="Hello! Introduce yourself in one sentence."
)

print(response.text)
print("\nUsage:", response.usage_metadata)

### 2. Load Documents & Build Search Index (`minsearch`)

In [ ]:
# Load documents
file_path = 'documents/all_documents.json'

with open(file_path, 'r', encoding='utf-8') as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents.")

# Index documents
index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)
print("Index fitted successfully!")

In [ ]:
def search(question, course='data-engineering', num_results=5):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course} if course else {}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

# Test search
sample_results = search('I just discovered the course. Can I join now?')
print(f"Found {len(sample_results)} results.")

### 3. Prompt Construction & System Instructions

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."
'''.strip()

USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''.strip()

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append(f"Q: {doc['question']}")
        lines.append(f"A: {doc['answer']}")
        lines.append('')
    return '\n'.join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    return USER_PROMPT_TEMPLATE.format(question=question, context=context)

### 4. Gemini LLM Function (with System Instruction & Config)

In [ ]:
def llm(instructions, user_prompt, model="gemini-3.7-flash", temperature=0.0):
    """
    Queries Gemini using Google GenAI SDK with system instruction.
    """
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=temperature
        )
    )
    return response.text

def llm_with_details(instructions, user_prompt, model="gemini-3.7-flash", temperature=0.0):
    """
    Queries Gemini and returns both text response and token usage metadata.
    """
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=temperature
        )
    )
    return {
        "answer": response.text,
        "usage": {
            "prompt_tokens": response.usage_metadata.prompt_token_count,
            "candidate_tokens": response.usage_metadata.candidates_token_count,
            "total_tokens": response.usage_metadata.total_token_count
        }
    }

### 5. End-to-End RAG Pipeline

In [ ]:
def rag(query, course='data-engineering', model="gemini-3.7-flash", num_results=5):
    search_results = search(query, course=course, num_results=num_results)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
# Example 1: Join question
q1 = "I just discovered the course. Can I join now?"
print(f"Q: {q1}")
print("A:", rag(q1))
print("-" * 60)

# Example 2: Certificate question
q2 = "How do I get a certificate?"
print(f"Q: {q2}")
print("A:", rag(q2))

### 6. Token Usage & Cost Calculation

In [ ]:
query = "How do I get a certificate?"
search_results = search(query, course='data-engineering')
prompt = build_prompt(query, search_results)

result = llm_with_details(INSTRUCTIONS, prompt)

# Gemini 2.5/3.7 Flash pricing (per 1M tokens approx: $0.10 input / $0.40 output)
input_price_per_1m = 0.10
output_price_per_1m = 0.40

prompt_tokens = result['usage']['prompt_tokens']
candidate_tokens = result['usage']['candidate_tokens']
cost = (prompt_tokens * input_price_per_1m / 1_000_000) + (candidate_tokens * output_price_per_1m / 1_000_000)

print("Answer:\n", result['answer'])
print("\n--- Token & Cost Stats ---")
print(f"Prompt Tokens:    {prompt_tokens}")
print(f"Candidate Tokens: {candidate_tokens}")
print(f"Total Tokens:     {result['usage']['total_tokens']}")
print(f"Estimated Cost:   ${cost:.6f}")

### 7. Multi-Turn Conversations / Chat in Gemini

Unlike OpenAI which uses `'developer'`, `'user'`, and `'assistant'`, Gemini uses `'user'` and `'model'` for turns, with `system_instruction` in the config.

In [ ]:
chat_history = [
    {"role": "user", "parts": [{"text": "Hi, which course is this?"}]},
    {"role": "model", "parts": [{"text": "This is the Data Engineering."}]},
    {"role": "user", "parts": [{"text": "Are there deadlines for the homeworks?"}]}
]

chat_response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents=chat_history,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS
    )
)

print("Model Response:", chat_response.text)